# 05 · E2: speed adaptation and speed–collision association (RQ2)

**RQ2:** Do speeds measured by WYS signs (85th-percentile speed and the share of speeding vehicles) decrease in adverse weather, and are locations with higher measured speeds associated with more collisions nearby?

* **Speed adaptation:** for sign *s* in hour *t*, V_st = α_s + β·W_t + δ·H_t, with a separate intercept per sign (α_s) and per hour of week (H_t). β therefore compares the same location under different weather. SEs are clustered by sign. The same model is fitted for the share of speeding vehicles.
* **Speed and collisions:** negative binomial regression of collisions near each sign on its 85th-percentile speed and posted limit, with log(active days) as exposure. Baseline: posted limit only. Buffers: 150 m and 250 m.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.spark_session import load_config, path, get_spark

cfg = load_config()
FIG, TAB = path(cfg, "figures"), path(cfg, "tables")
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
PROC = path(cfg, "processed"); PROC.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
from src import models as M
spark = get_spark(cfg); spark.sparkContext.setLogLevel("ERROR")
cols = ["sign_id", "date", "hour", "volume", "p85_speed", "share_speeding", "mean_speed", "speed_limit",
        "rain", "snow", "vis_lt1", "vis_1to5", "snow_on_ground", "temp_bin", "hour_of_week", "year"]
sh = spark.read.parquet(str(PROC / "sign_hour_panel")).select(*cols).toPandas()
spark.stop()
sh = sh[sh["temp_bin"] != "nan"]
temp = pd.get_dummies(sh["temp_bin"], prefix="temp", dtype=float).drop(columns="temp_10_20")
sh = pd.concat([sh, temp], axis=1)
print(f"{len(sh):,} sign-hours, {sh['sign_id'].nunique()} signs")

## 1. Speed adaptation (sign and hour-of-week fixed effects)

In [ ]:
x = ["rain", "snow", "vis_lt1", "vis_1to5", "snow_on_ground"] + list(temp.columns)
p85, info = M.fe_regression(sh, "p85_speed", x)
share, _ = M.fe_regression(sh.assign(share_pct=100 * sh["share_speeding"]), "share_pct", x)
print(info)
out = pd.concat({"85th-percentile speed (km/h)": p85, "share speeding (percentage points)": share}, axis=0)
out.to_csv(TAB / "e2_speed_fe.csv")
out.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
names = {"rain": "rain", "snow": "snow", "vis_lt1": "vis < 1 km", "vis_1to5": "vis 1-5 km", "snow_on_ground": "snow on ground"}
for a, (t, tab, unit) in zip(ax, [("85th-percentile speed", p85, "km/h"), ("Share speeding", share, "percentage points")]):
    tt = tab.loc[list(names)].iloc[::-1]
    a.errorbar(tt["estimate"], range(len(tt)), xerr=[tt["estimate"] - tt["ci_low"], tt["ci_high"] - tt["estimate"]], fmt="o", capsize=3, color="#55A868")
    a.axvline(0, color="grey", ls="--", lw=0.8); a.set_yticks(range(len(tt))); a.set_yticklabels([names[i] for i in tt.index])
    a.set_title(t); a.set_xlabel(f"change vs dry, clear hour ({unit})")
fig.savefig(FIG / "fig_e2_speed_change.png"); plt.show()

In [ ]:
# Baseline comparison: within-sign variance explained by weather (model without weather explains 0 by construction)
d = sh[["p85_speed"] + x + ["sign_id", "hour_of_week"]].dropna()
dm = M.twoway_demean(d, ["p85_speed"] + x, "sign_id", "hour_of_week")
import statsmodels.api as sm
r2 = sm.OLS(dm["p85_speed"], dm[x]).fit().rsquared
print(f"within R-squared added by weather: {r2:.4f}")

## 2. Speed and nearby collisions (one row per sign)

In [ ]:
import statsmodels.formula.api as smf
from scipy import stats
rows, tabs = [], {}
for r in cfg["spatial"]["e2_buffers_m"]:
    st = pd.read_parquet(PROC / f"sign_level_{r}m.parquet")
    st = st[(st["active_days"] > 0) & st["p85_speed"].notna() & st["speed_limit"].notna()].copy()
    st["limit"] = st["speed_limit"].astype(int).astype(str)
    st["exposure"] = st["active_days"]
    full = M.fit_nb("collisions ~ p85_speed + C(limit)", st, np.log(st["exposure"]))
    base = M.fit_nb("collisions ~ C(limit)", st, np.log(st["exposure"]))
    lr = 2 * (full.llf - base.llf)
    t = M.ratio_table(full, exclude_prefixes=("Intercept", "alpha"))
    tabs[r] = t
    rows.append({"buffer_m": r, "signs": len(st), "collisions": int(st["collisions"].sum()),
                 "IRR per km/h": t.loc["p85_speed", "estimate"], "ci_low": t.loc["p85_speed", "ci_low"],
                 "ci_high": t.loc["p85_speed", "ci_high"], "LR vs limit-only": lr, "p": stats.chi2.sf(lr, 1),
                 "AIC limit-only": base.aic, "AIC with speed": full.aic})
e2b = pd.DataFrame(rows)
e2b.to_csv(TAB / "e2_speed_collisions.csv", index=False)
display(e2b.round(4))
pd.concat(tabs).round(3)

In [ ]:
st = pd.read_parquet(PROC / f"sign_level_{cfg['spatial']['buffer_m']}m.parquet")
st = st[st["active_days"] > 0]
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for lim, g in st.groupby("speed_limit"):
    ax.scatter(g["p85_speed"], 100 * g["collisions"] / g["active_days"], s=10, alpha=0.6, label=f"{int(lim)} km/h")
ax.set_xlabel("85th-percentile speed (km/h)"); ax.set_ylabel("collisions within 150 m per 100 active days")
ax.set_title("WYS sites: measured speed and nearby collisions"); ax.legend(title="posted limit", frameon=False, fontsize=7)
fig.savefig(FIG / "fig_e2_speed_vs_collisions.png"); plt.show()

## 3. Hypothesis H2

In [ ]:
h2 = p85.loc[["rain", "snow"], ["estimate", "ci_low", "ci_high"]].copy()
h2.index = ["rain: change in 85th-pct speed (km/h)", "snow: change in 85th-pct speed (km/h)"]
s2 = share.loc[["rain", "snow"], ["estimate", "ci_low", "ci_high"]].copy()
s2.index = ["rain: change in share speeding (pp)", "snow: change in share speeding (pp)"]
h2 = pd.concat([h2, s2])
h2["verdict (expected < 0)"] = [M.hypothesis_verdict(r.estimate, r.ci_low, r.ci_high, null=0, expected="less") for r in h2.itertuples()]
h2.to_csv(TAB / "h2.csv")
h2.round(3)